# Band broadening in tube flow — N stirred tanks in series
### A self-contained Colab companion to the `tube` Streamlit app

A tube of mean residence time $\tau$ is approximated by $N$ equal perfectly mixed
tanks in series. A rectangular tracer pulse ($C_p$ on $[0, t_p]$) is fed to tank 1.
We integrate the $N$ tank balances with **diffrax** (JAX), obtain the **Jacobian by
automatic differentiation** (`jax.jacfwd`), exploit its bidiagonal sparsity in the
implicit solver, and verify everything against exact theory.

**You will see:** the model, AD-discovered sparse Jacobian, diffrax integration,
exact gamma-CDF check, band snapshots, the $N\to\infty$ plug-flow limit, moment
checks, implicit-vs-explicit stiffness demo, and a dense-vs-sparse scaling race
(stopping once either run exceeds 30 s).

In [ ]:
!pip install -q diffrax plotly   # tested with jax 0.11.2, diffrax 0.7.2, lineax 0.1.1, equinox 0.13.8

import jax, diffrax, equinox as eqx
import jax.numpy as jnp
import lineax as lx
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

jax.config.update("jax_enable_x64", True)  # tight tolerances need float64
print("jax", jax.__version__, "| diffrax", diffrax.__version__)

## 1. Model setup

Tube volume $V$, steady flow $Q$ → mean residence time $\tau = V/Q$.
Replace the tube by $N$ equal CSTRs, each of residence time $\tau/N$.
Assumptions: incompressible, constant $Q$, passive tracer (no reaction/adsorption),
isothermal, each tank perfectly mixed. Feed: rectangular pulse
$C_0(t) = C_p$ on $[0,t_p]$, $0$ otherwise. Injected mass $M = Q\,C_p\,t_p$.

In [ ]:
TAU, CP, TP = 10.0, 1.0, 1.0   # app defaults
N_EFF  = [1, 3, 12, 50]          # effluent comparison set
N_SNAP = 12                      # band snapshots
N_PF   = 150                     # plug-flow limit test (app default)

def make_rhs(N, tau, Cp, tp, feed_on):
    """dC_i/dt = (N/tau)(C_{i-1} - C_i), C_0(t) = rectangular pulse."""
    def rhs(t, y, args):
        pulse = jnp.where(feed_on & (t >= 0.0) & (t <= tp), Cp, 0.0)
        cin = jnp.empty_like(y).at[0].set(pulse).at[1:].set(y[:-1])
        return (N / tau) * (cin - y)
    return rhs

## 2. Component balance

Unsteady tracer balance on tank $i$ (in $-$ out $=$ accumulation):

$$\frac{d}{dt}(V_i C_i) = Q\,(C_{i-1} - C_i) \;\;\Longrightarrow\;\;
\boxed{\frac{dC_i}{dt} = \frac{N}{\tau}\,(C_{i-1} - C_i)},\quad i=1\ldots N$$

i.e. $\dot{\mathbf C} = \tfrac{N}{\tau}(S\mathbf C + \mathbf e_1 C_0(t))$
with $S_{ii}=-1$, $S_{i,i-1}=+1$. The RHS is built vectorized in $O(N)$ —
`pulse` feeds row 0, every other row sees the previous tank — without forming $S$.

## 3. Jacobian by automatic differentiation

The system is **linear with constant coefficients**, so its Jacobian is constant.
Rather than hand-coding it, we let AD discover it — and its sparsity:

$$J = \frac{\partial f}{\partial \mathbf C}
   = \frac{N}{\tau}\begin{pmatrix}
     -1&&&&\\\\ 1&-1&&&\\\\ &1&-1&&\\\\ &&\ddots&\ddots&\\\\ &&&1&-1
     \end{pmatrix}$$

`jax.jacfwd` differentiates the *vectorized* RHS directly. Below we assert
constancy (same $J$ at two different $(t,\mathbf C)$) and the bidiagonal pattern —
this discovered structure is what the sparse solver below relies on.

In [ ]:
N = 12
rhs = make_rhs(N, TAU, CP, TP, True)

J       = jax.jacfwd(rhs, argnums=1)(0.0, jnp.zeros(N), None)
J_other = jax.jacfwd(rhs, argnums=1)(3.7, jnp.full(N, 2.5), None)
assert jnp.allclose(J, J_other), "Jacobian is not constant!"

d, l = jnp.diag(J), jnp.diag(J, -1)
assert jnp.allclose(J, jnp.diag(d) + jnp.diag(l, -1)), "Jacobian is not bidiagonal!"
assert jnp.allclose(d, -N / TAU) and jnp.allclose(l, N / TAU)  # exact values
print(f"J is constant and bidiagonal: diag={d[0]:.3f}, subdiag={l[0]:.3f} (= ∓N/τ)")

fig = go.Figure(go.Heatmap(z=(np.asarray(J) != 0).astype(int),
                           colorscale=[[0, "white"], [1, "#1f77b4"]],
                           showscale=False))
fig.update_layout(title="Sparsity pattern of the AD Jacobian (N=12)",
                  xaxis_title="column", yaxis_title="row",
                  yaxis_autorange="reversed", width=420, height=420)
fig.show()

## 4. Integration with diffrax

`diffrax.Kvaerno5` (5th-order ESDIRK — the diffrax analog of the app's Radau) solves
each implicit stage with `VeryChord`, which linearizes the stage equations **once per
step by AD** and reuses that Jacobian for the chord iterations (a linear problem converges in one Newton step; diffrax's `VeryChord`
still enforces at least two chord iterations before checking termination). The linear system of every stage is
$(I - h\\,a_{ii}J)\\,x = b$ — still bidiagonal — so we hand diffrax a tiny
`lineax` solver doing $O(N)$ Thomas substitution instead of dense $O(N^3)$ LU:

* `init` extracts the two bands from the operator diffrax actually passes, and
  **fails loudly** (trace-safe `eqx.error_if`) if it isn't lower-bidiagonal;
* `compute` runs the branch-free Thomas algorithm, which also covers the
  transposed (upper-bidiagonal) system.

One API note: diffrax 0.7 exposes no `jump_ts`/discontinuity hook, so — as in the
app — we integrate the pulse-on phase $[0,t_p]$ and pulse-off phase $[t_p,t_{end}]$
separately and stitch at $t_p$ (never integrating across the kink).

In [ ]:
class BidiagonalSolver(lx.AbstractLinearSolver):
    """O(N) Thomas solver for (lower-)bidiagonal systems, for diffrax stages."""
    def init(self, operator, options):
        A = operator.as_matrix()
        d, l = jnp.diag(A), jnp.diag(A, -1)
        resid = jnp.max(jnp.abs(A - (jnp.diag(d) + jnp.diag(l, -1))))
        A = eqx.error_if(A, resid > 1e-10 * jnp.max(jnp.abs(A)),
                         "BidiagonalSolver: stage matrix is not lower-bidiagonal")
        return (d, l, jnp.zeros_like(l))  # (diag, sub, super)

    def _solve(self, d, sub, sup, b):
        if d.shape[0] == 1:  # shapes are static: trace-safe N=1 shortcut
            return b / d
        def fwd(carry, i):  # eliminate subdiagonal
            dm1, bm1 = carry
            w = sub[i - 1] / dm1
            di, bi = d[i] - w * sup[i - 1], b[i] - w * bm1
            return (di, bi), (di, bi)
        (_, _), (ds, bs) = jax.lax.scan(fwd, (d[0], b[0]), jnp.arange(1, len(d)))
        dall = jnp.concatenate([d[:1], ds])
        ball = jnp.concatenate([b[:1], bs])
        def bwd(x_next, i):  # back substitution
            xi = (ball[i] - sup[i] * x_next) / dall[i]
            return xi, xi
        _, xs = jax.lax.scan(bwd, ball[-1] / dall[-1],
                             jnp.arange(len(d) - 2, -1, -1))
        return jnp.concatenate([xs[::-1], ball[-1:] / dall[-1:]])

    def compute(self, state, vector, options):
        d, sub, sup = state
        return self._solve(d, sub, sup, vector), lx.RESULTS.successful, {}

    def transpose(self, state, options):
        d, sub, sup = state
        return (d, sup, sub), options

    def conj(self, state, options):
        d, sub, sup = state
        return (d.conj(), sub.conj(), sup.conj()), options

    def assume_full_rank(self):
        return True


def kvaerno5_sparse():
    return diffrax.Kvaerno5(
        root_finder=diffrax.with_stepsize_controller_tols(diffrax.VeryChord)(
            linear_solver=BidiagonalSolver()))

DENSE_SOLVER = diffrax.Kvaerno5()   # default: AD Jacobian + dense LU
SPARSE_SOLVER = kvaerno5_sparse()   # same method, sparse banded linear algebra


def simulate(N, solver, n=1200, tail=5.0):
    """Two-phase diffrax solve; returns (t, C) with C[k, i] = tank i+1 at t[k]."""
    t_end = TAU + TP + tail * TAU          # 5*tau of tail: N=1 tail down to e^-5
    t_eval = jnp.linspace(0.0, t_end, n)
    t_eval = jnp.sort(jnp.unique(jnp.concatenate([t_eval, jnp.array([0.0, TP])])))
    i_tp = int(jnp.searchsorted(t_eval, TP))
    t_a, t_b = t_eval[:i_tp + 1], t_eval[i_tp:]
    ctrl = diffrax.PIDController(rtol=1e-8, atol=1e-10)
    kw = dict(args=None, stepsize_controller=ctrl, max_steps=500_000)
    sol_a = diffrax.diffeqsolve(diffrax.ODETerm(make_rhs(N, TAU, CP, TP, True)),
                                solver, t0=0.0, t1=TP, dt0=TP / 50, y0=jnp.zeros(N),
                                saveat=diffrax.SaveAt(ts=t_a), **kw)
    sol_b = diffrax.diffeqsolve(diffrax.ODETerm(make_rhs(N, TAU, CP, TP, False)),
                                solver, t0=TP, t1=t_b[-1], dt0=TP / 50,
                                y0=sol_a.ys[-1],
                                saveat=diffrax.SaveAt(ts=t_b), **kw)
    t = np.concatenate([t_a, t_b[1:]])
    C = np.concatenate([sol_a.ys, sol_b.ys[1:]], axis=0)
    return t, C
print("solvers ready")

## 5. Exact response to the rectangular pulse

The RTD of $N$ tanks in series follows from the Laplace transfer function
$G_N(s) = (1+\tau s/N)^{-N}$ — the Erlang (gamma) distribution

$$\boxed{E(t) = \\frac{(N/\\tau)^N}{(N-1)!}\\,t^{N-1}e^{-Nt/\\tau}},\\qquad
\\bar t = \\tau,\\;\\; \\sigma^2 = \\tau^2/N.$$

Convolving the rectangular feed with $E(t)$ gives a difference of two gamma CDFs
$F$ (shape $N$, scale $\tau/N$):

$$\boxed{C_N(t) = C_p\\,[F(t) - F(t-t_p)]}$$

— the independent check on the numerics. Variances add under convolution:
$\sigma^2_{\\mathrm{effluent}} = \\tau^2/N + t_p^2/12$.

In [ ]:
def exact_pulse_response(t, N, tau, Cp, tp):
    F = jax.scipy.stats.gamma.cdf(t, N, scale=tau / N)
    return Cp * (F - jax.scipy.stats.gamma.cdf(t - tp, N, scale=tau / N))

fig = make_subplots(rows=2, cols=2, subplot_titles=[f"N = {N}" for N in N_EFF],
                    x_title="t", y_title="C_N(t)")
for k, N in enumerate(N_EFF):
    t, C = simulate(N, SPARSE_SOLVER)
    te = np.linspace(0, t[-1], 2000)
    fig.add_scatter(x=t, y=C[:, -1], mode="lines", name=f"num N={N}",
                    legendgroup=k, showlegend=(k == 0),
                    row=k // 2 + 1, col=k % 2 + 1)
    fig.add_scatter(x=te, y=np.asarray(exact_pulse_response(te, N, TAU, CP, TP)),
                    mode="lines", line=dict(dash="dash"), name=f"exact N={N}",
                    legendgroup=k, showlegend=(k == 0),
                    row=k // 2 + 1, col=k % 2 + 1)
    dev = np.max(np.abs(C[:, -1] - np.asarray(exact_pulse_response(t, N, TAU, CP, TP))))
    print(f"N={N:3d}: max |numerical - exact| = {dev:.2e}")
    assert dev < 1e-6, f"exact check failed for N={N}"
fig.update_layout(title="Effluent: diffrax (solid) vs exact gamma-CDF (dashed)",
                  height=620)
fig.show()

## 6. Band propagation

Snapshots of the band $C(\\mathrm{tank})$ at times centered on the band
($\\approx\\tau + t_p/2$) and spaced one band-width ($2\\sigma = 2\\tau/\\sqrt N$)
apart:

In [ ]:
t, C = simulate(N_SNAP, SPARSE_SOLVER)
centers = TAU + TP / 2 + np.arange(-2, 3) * (2 * TAU / np.sqrt(N_SNAP))
t_snap = np.clip(centers, 0.5, t[-1])
tanks = np.arange(1, N_SNAP + 1)

fig = go.Figure()
for ts in t_snap:
    k = int(np.argmin(np.abs(t - ts)))
    fig.add_scatter(x=tanks, y=C[k], mode="lines+markers", name=f"t = {t[k]:.1f}")
fig.update_layout(title=f"Band snapshots, N = {N_SNAP}",
                  xaxis_title="tank", yaxis_title="C")
fig.show()

## 7. Plug-flow limit ($N \\to \\infty$)

$\\lim_{N\\to\\infty}(1+\\tau s/N)^{-N} = e^{-\\tau s}$ — a **pure delay** $\\tau$.
The effluent tends to the feed shifted by $\\tau$ with zero broadening:

In [ ]:
t, C = simulate(N_PF, SPARSE_SOLVER)
rect = np.where((t >= TAU) & (t <= TAU + TP), CP, 0.0)
# like the app's test: compare away from the edge boundary layers
w = 4.0 * TAU / np.sqrt(N_PF) + 0.5 * TP
mask = (np.abs(t - TAU) > w) & (np.abs(t - (TAU + TP)) > w)
dev = np.max(np.abs(C[:, -1][mask] - rect[mask]))
print(f"N={N_PF}: max |num - plug-flow| away from edges = {dev:.2e} (Cp = {CP})")
assert dev < 0.03 * CP, "plug-flow collapse failed"

fig = go.Figure()
fig.add_scatter(x=t, y=C[:, -1], mode="lines", name=f"numerical N={N_PF}")
fig.add_scatter(x=t, y=rect, mode="lines", line=dict(dash="dash"),
                name="plug flow: Cp on [τ, τ+tp]")
fig.update_layout(title=f"Plug-flow limit at N = {N_PF}",
                  xaxis_title="t", yaxis_title="C_N(t)")
fig.show()

## 8. Moment checks

Integral checks on every numerical effluent curve (trapezoidal rule):

$$\\int C_N\\,dt = C_p t_p,\\qquad
\\bar t = \\frac{\\int t C_N dt}{\\int C_N dt} = \\tau + \\frac{t_p}{2},\\qquad
\\sigma^2 = \\frac{\\int t^2 C_N dt}{\\int C_N dt} - \\bar t^2
          = \\frac{\\tau^2}{N} + \\frac{t_p^2}{12}.$$

Any violation flags a numerical or modelling error:

In [ ]:
def moments(t, c):
    m0 = np.trapezoid(c, t)
    mean = np.trapezoid(t * c, t) / m0
    var = np.trapezoid(t**2 * c, t) / m0 - mean**2
    return m0, mean, var

N_grid = [1, 2, 3, 5, 8, 12, 20, 35, 50, 80, 120]
sig2_num, sig2_th = [], []
for N in N_grid:
    t, C = simulate(N, SPARSE_SOLVER, tail=10.0)  # long tail captures N=1 mass
    m0, mean, var = moments(t, C[:, -1])
    assert abs(m0 - CP * TP) / (CP * TP) < 1e-3, f"mass failed at N={N}"
    assert abs(mean - (TAU + TP / 2)) < 5e-3, f"mean failed at N={N}"
    sig2_num.append(var)
    sig2_th.append(TAU**2 / N + TP**2 / 12)
    assert abs(var - sig2_th[-1]) / sig2_th[-1] < 5e-3, f"variance failed at N={N}"
print("mass, mean, variance checks passed for all N")

fig = go.Figure()
fig.add_scatter(x=1 / np.array(N_grid), y=sig2_num, mode="lines+markers",
                name="numerical σ²")
fig.add_scatter(x=1 / np.array(N_grid), y=sig2_th, mode="lines",
                line=dict(dash="dash"), name="τ²/N + tₚ²/12")
fig.update_layout(title="Effluent variance → plug flow as 1/N → 0",
                  xaxis_title="1/N", yaxis_title="σ²",
                  xaxis_type="log", yaxis_type="log")
fig.show()

## 9. Stiffness: implicit vs explicit

The fastest mode decays at rate $N/\\tau$ ($20$ at $N=200$) — mildly stiff.
An explicit method (Tsit5) is stability-limited to $\\Delta t \\lesssim \\tau/N$,
so its step count grows with $N$; Kvaerno5 steps over the fast transient.
(Per-step cost is a separate matter — an explicit step is just a few RHS
evaluations, while an implicit step solves Newton systems — so wall time and
step count tell different parts of the story.)

In [ ]:
import time

def solve_stats(N, solver):
    t_end = TAU + TP + 5.0 * TAU
    tg = jnp.sort(jnp.unique(jnp.concatenate(
        [jnp.linspace(0.0, t_end, 1200), jnp.array([0.0, TP])])))
    i_tp = int(jnp.searchsorted(tg, TP))
    t_a, t_b = tg[:i_tp + 1], tg[i_tp:]
    ctrl = diffrax.PIDController(rtol=1e-8, atol=1e-10)
    def run():
        sa = diffrax.diffeqsolve(diffrax.ODETerm(make_rhs(N, TAU, CP, TP, True)),
                                 solver, t0=0.0, t1=TP, dt0=TP / 50,
                                 y0=jnp.zeros(N), args=None,
                                 saveat=diffrax.SaveAt(ts=t_a),
                                 stepsize_controller=ctrl, max_steps=5_000_000)
        sb = diffrax.diffeqsolve(diffrax.ODETerm(make_rhs(N, TAU, CP, TP, False)),
                                 solver, t0=TP, t1=t_b[-1], dt0=TP / 50,
                                 y0=sa.ys[-1], args=None,
                                 saveat=diffrax.SaveAt(ts=t_b),
                                 stepsize_controller=ctrl, max_steps=5_000_000)
        jax.block_until_ready((sa.ys, sb.ys))
        return int(sa.stats["num_steps"] + sb.stats["num_steps"])
    run()  # warmup
    t0 = time.perf_counter(); steps = run(); dt = time.perf_counter() - t0
    return steps, dt

for N in [100, 200, 400]:
    se, dte = solve_stats(N, diffrax.Tsit5())
    si, dti = solve_stats(N, SPARSE_SOLVER)
    print(f"N={N}: Tsit5 {se} steps / {dte:.2f} s | "
          f"Kvaerno5-sparse {si} steps / {dti:.2f} s")
    assert si < se, "implicit should need fewer steps (stiffness signature)"
print("stiffness signature confirmed: explicit step count grows ~proportionally to N")

## 10. The sparse-Jacobian payoff: dense vs sparse scaling

Same method (Kvaerno5), same tolerances — the only difference is the linear algebra
inside the Newton iteration: dense LU vs the $O(N)$ bidiagonal Thomas solver built
from the AD-discovered Jacobian. Doubling $N$ until either run exceeds **30 s**:

In [ ]:
import time

def solve_timed(N, solver):
    def run():
        t, C = simulate(N, solver)
        jax.block_until_ready(C)
        return t, C
    run()  # warmup: exclude JIT compilation
    t0 = time.perf_counter()
    t, C = run()
    return time.perf_counter() - t0, C

results = []
N = 100
while True:
    row = {"N": N}
    for name, solver in [("dense", DENSE_SOLVER), ("sparse", SPARSE_SOLVER)]:
        dt, _ = solve_timed(N, solver)
        row[name] = dt
        print(f"N={N:<6d} {name:<6s} {dt:7.2f} s", flush=True)
    results.append(row)
    if max(row["dense"], row["sparse"]) > 30.0:
        print(f"30 s cap reached at N = {N}")
        break
    N *= 2

fig = go.Figure()
for name in ["dense", "sparse"]:
    fig.add_scatter(x=[r["N"] for r in results], y=[r[name] for r in results],
                    mode="lines+markers", name=name)
fig.update_layout(title="Wall time vs N: dense LU vs sparse bidiagonal (Kvaerno5)",
                  xaxis_title="N", yaxis_title="solve time (s)",
                  xaxis_type="log", yaxis_type="log")
fig.show()

## What we kept from the app — and what changed

Kept: the model, all seven theory sections (condensed), the exact gamma-CDF check,
plug-flow limit, moment checks, and the app's default parameters
($\\tau=10$, $C_p=1$, $t_p=1$).
Changed: `scipy` → **diffrax** + JAX; the Jacobian comes from **`jax.jacfwd`**
(constant, bidiagonal — asserted, not hand-coded) and its sparsity is exploited
by a custom banded linear solver inside the implicit Newton iteration; the pulse
kink is handled by two stitched phases (diffrax 0.7 has no `jump_ts` hook).